In [ ]:
import sys
sys.path.insert(0, "../../../")

from src.agents import AgentDriver, AgentDriverConfig
from src.agents.utils import AgentConnectorConfig

import json
import re
from pydantic import BaseModel, Field
from typing import List, Literal
from langchain_core.output_parsers import JsonOutputParser
from langchain_core.exceptions import OutputParserException
from langchain_core.prompts import PromptTemplate
from collections import defaultdict
from tqdm import tqdm

In [ ]:
BENCHMARK = "hotpotqa_distractor_validation" # TO CHANGE
FORMATTED_TRACES_PATH = f"../../../data/experiences_extraction/formatted_traces/{BENCHMARK}/without_queryprep/part1(casual_answer_generator).json"

In [ ]:
QUERIES_ANALYSIS_SAVE_PATH = f"../../../data/experiences_extraction/general_extraction(iteration#1)/{BENCHMARK}/queries_analysis.json"

In [ ]:
with open(FORMATTED_TRACES_PATH, 'r', encoding='utf-8') as fd:
    traces = json.loads(fd.read())

In [ ]:
agentd_config = AgentDriverConfig(
    agent_config=AgentConnectorConfig(
        credentials={ 'model': 'gemma3:12b','host': 'localhost','port': 11437 }
    )
)

In [ ]:
agent = AgentDriver.connect(agentd_config)

In [ ]:
class QueryAnalysis(BaseModel):
    formalized_query: str = Field(
        description="Clear and full formulation of the given query and its key parameters."
    )
    specialized_tags: List[str] = Field(
        description="Relevant keywords from the 'formalized_query' field: nouns, verbs, and technical terms."
    )
    specialized_intents: List[str] = Field(
        description="List of specific/detailed intents that is explicitly or implicitly stated in a 'formalized_query' field: detailed actions such as 'retrieve latest version', 'compare specifications', 'calculate ROI'."
    )
    masked_query: str = Field(
        description="Formalized query with masked named entities and specific terminology. For masking use '[Named Entity]' or '[General Term]' keywords."
    )
    generalized_query: str = Field(
        description="Abstraction of the formalized query to create a more generalized task representation."
    )
    generalized_intents: List[str] = Field(
        description="List of general/broad intents that is explicitly or implicitly stated in a 'generalized_query': broader categories like 'information retrieval', 'decision support', 'knowledge extraction'."
    )
    generalized_tags: List[str] = Field(
        description="Relevant keywords from the 'generalized_query' field: high‑level concepts (e.g., 'performance', 'pricing', 'legal compliance')."
    )

In [ ]:
QUERY_ANALYSIS_SYSTEM_PROMPT = '''
You are an expert AI analyst reviewing user queries that is going to LLM-based search engines.
Your task is to extract reusable, actionable experiences that can guide future agent executions.
Focus on identifying specific patterns, techniques, and decision points that contributed to success.
ANALYSIS FRAMEWORK:
* UNDERSTANDING THE USER GOAL: Identify the primary objective (e.g., "find", "compare", "summarize", "explain"). Distinguish between explicit requests and implicit needs.
* PARAMETER EXTRACTION: Detect concrete parameters (dates, quantities, locations, product/model numbers, etc.). Preserve units and qualifiers (e.g., “> 10 GB”, “last 3 months”).
* REUSABILITY: Extract patterns that can be applied to similar scenarios.
* FORMALIZATION RULES: Expand abbreviations, resolve pronouns, and make implicit context explicit. Keep the wording concise but complete; avoid conversational filler.

# Original Query
{query}

{output_format}
'''

In [ ]:
queryanalysis_parser = JsonOutputParser(pydantic_object=QueryAnalysis)
print(queryanalysis_parser.get_format_instructions())

In [ ]:
queryanalysis_template = PromptTemplate(
    template=QUERY_ANALYSIS_SYSTEM_PROMPT,
    input_variables=["query"], # Что должен предоставить пользователь
    partial_variables={         # Что заполняется автоматически
        "output_format": queryanalysis_parser.get_format_instructions()
    }
)

In [ ]:
print(len(list(traces.keys())))

In [ ]:
queries_info = dict()
max_trials = 3
parsing_errors_counter = 0
for i, query_id in enumerate(list(traces.keys())):
    print(f"{i}. query_id = {query_id}; query = '{traces[query_id]['question']}'")
    formatted_system_prompt = queryanalysis_template.format(
        query=traces[query_id]['question']
    )
    current_trial = 1
    is_output_parsed = False
    while current_trial <= max_trials:
        raw_response, stat = agent.generate(system_prompt=formatted_system_prompt, assistant_prompt="Query Analysis:\n")
        try:
            parsed_response = queryanalysis_parser.invoke(raw_response)
        except OutputParserException as e:
            current_trial += 1
        else:
            is_output_parsed = True
            break

    if not is_output_parsed:
        print(f"!!! Parsing Error: query_id = {query_id} !!!")
        print(raw_response)
        queries_info[query_id] = None
        parsing_errors_counter += 1
        continue
    else:
        print(json.dumps(parsed_response, ensure_ascii=False, indent=2))
        queries_info[query_id] = parsed_response

    print("="*10)

In [ ]:
with open(QUERIES_ANALYSIS_SAVE_PATH, 'w', encoding='utf-8') as fd:
    fd.write(json.dumps(queries_info, ensure_ascii=False, indent=2))

In [ ]:
print(parsing_errors_counter)

In [ ]:
agent.close_connection()